# 06 — Simulation: Policy Runner & Recording

`run_policy()` orchestrates the full sim loop. Domain randomization
and LeRobotDataset recording are built-in.

In [ ]:
from strands_robots.simulation import create_simulation

sim = create_simulation()
sim.create_world(timestep=0.002)
sim.add_robot("so100", data_config="so100", position=[0, 0, 0])
sim.add_object("cube", shape="box", position=[0.3, 0, 0.05],
               size=[0.03, 0.03, 0.03], color=[1, 0, 0, 1])
print("world ready ✅")

## `run_policy()` — obs → policy → action → step

In [ ]:
result = sim.run_policy(
    robot_name="so100",
    policy_provider="mock",
    instruction="pick up the red cube",
    duration=2.0,           # seconds of sim time
    record_video=None,      # set to "output.mp4" to save video
)
print(result["content"][0]["text"])

In [ ]:
# Visualize the scene after policy execution
from IPython.display import display, Image as IPImage

result = sim.render(camera_name="default", width=640, height=480)
if result.get("status") == "success":
    for item in result.get("content", []):
        if "image" in item:
            display(IPImage(data=item["image"]["source"]["bytes"]))
            break
    print(result["content"][0]["text"])
else:
    print(result["content"][0]["text"])

## Domain Randomization (RandomizationMixin)

In [ ]:
result = sim.randomize(
    randomize_colors=True,
    randomize_lighting=True,
    randomize_physics=True,
    randomize_positions=True,
    position_noise=0.02,
    seed=42,
)
print(result["content"][0]["text"])

In [ ]:
# Render after randomization — colors, lighting, positions changed
result = sim.render(camera_name="default", width=640, height=480)
if result.get("status") == "success":
    for item in result.get("content", []):
        if "image" in item:
            display(IPImage(data=item["image"]["source"]["bytes"]))
            break
    print("After randomization ↑")

## LeRobotDataset Recording (RecordingMixin)

```python
sim.start_recording(
    repo_id="user/my_sim_dataset",
    task="pick up cube",
    fps=30,
    vcodec="libsvtav1",
)

sim.run_policy("so100", "mock", instruction="pick up cube", duration=5.0)

sim.stop_recording()
# or: sim.stop_recording(push_to_hub=True)
```

## Evaluation Loop Pattern

```python
success_count = 0
n_episodes = 10

for ep in range(n_episodes):
    sim.reset()
    sim.randomize(seed=ep)

    sim.run_policy("so100", "mock", instruction="pick up cube", duration=5.0)

    obs = sim.get_observation("so100")
    # success = check_success(obs)
    # if success: success_count += 1

print(f"Success rate: {success_count}/{n_episodes}")
```

In [ ]:
sim.destroy()
print("done ✅")